# Evaluate similarities

### Confirm the directories are correct

In [ ]:
import os
from pathlib import Path
import json

print('Notebook dir:', os.getcwd())
roots = [Path('../data/processed/kc'), Path('../data/processed/portland')]
for r in roots:
    print(f"Root: {r} -> exists: {r.exists()}")
    if r.exists():
        files = list(r.rglob('*.jsonl'))
        print(f"  jsonl files: {len(files)}")
        if files:
            sample = files[0]
            print('  sample file:', sample)
            with sample.open('r', encoding='utf-8', errors='ignore') as fh:
                for i, line in enumerate(fh):
                    if not line.strip():
                        continue
                    try:
                        obj = json.loads(line)
                        print('   first keys:', list(obj.keys())[:10])
                        print('   first snippet:', line.strip()[:200])
                    except Exception as e:
                        print('   first line not JSON:', line.strip()[:200])
                    break


## Find all of the groups of sentences where they are different by only one word or number

In [7]:
from pathlib import Path
import re
import json
from collections import defaultdict
import pandas as pd

roots = [Path("../data/processed/kc"), Path("../data/processed/portland")]
skip_dir_names = {"stats"}

source_col_candidates = {
    "source", "src", "english_source", "en_source", "english", "sentence", "text", "source_sentence"
}

# Filtering flags: change these to control short-sentence exclusion
# Set `exclude_short` to False to keep one-word sentences (e.g., headings)
EXCLUDE_SHORT = True
MIN_WORDS = 2

def tokenize(s):
    return re.findall(r"\d+(?:\.\d+)?|\w+|[^\w\s]", str(s).strip().lower())

def _extract_strings(v):
    res = []
    if isinstance(v, str):
        if v.strip():
            res.append(v.strip())
        return res
    if isinstance(v, dict):
        for k in ("eng", "en", "english"):
            if k in v and isinstance(v[k], str) and v[k].strip():
                return [v[k].strip()]
        for val in v.values():
            if isinstance(val, str) and val.strip():
                res.append(val.strip())
            elif isinstance(val, (list, dict)):
                res.extend(_extract_strings(val))
        return res
    if isinstance(v, list):
        for item in v:
            res.extend(_extract_strings(item))
        return res
    return res


def _extract_from_object(obj):
    if not isinstance(obj, dict):
        return []
    # prefer explicit `source`
    if "source" in obj:
        vals = _extract_strings(obj["source"])
        if vals:
            return vals
    # prefer translations.en / translations.eng
    if "translations" in obj and isinstance(obj["translations"], dict):
        for k in ("eng", "en", "english"):
            if k in obj["translations"] and isinstance(obj["translations"][k], str) and obj["translations"][k].strip():
                return [obj["translations"][k].strip()]
    # known source-like keys
    for k in obj.keys():
        if k.lower() in source_col_candidates:
            vals = _extract_strings(obj[k])
            if vals:
                return vals
    # fallback: any nested string
    for v in obj.values():
        vals = _extract_strings(v)
        if vals:
            return vals
    return []

def _word_count(s):
    return len(re.findall(r"\w+", str(s)))


def read_sentences_from_file(path):
    sentences = []
    suffix = path.suffix.lower()

    # Only support .jsonl for this analysis
    if suffix == ".jsonl":
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except Exception:
                    continue
                vals = _extract_from_object(obj)
                if vals:
                    sentences.extend(vals)

    return sentences

all_sentences = []
existing_roots = [r for r in roots if r.exists()]
for root in existing_roots:
    for path in root.rglob("*"):
        if not path.is_file():
            continue
        if any(part in skip_dir_names for part in path.parts):
            continue
        all_sentences.extend(read_sentences_from_file(path))

# Deduplicate and optionally exclude short sentences
seen = set()
unique_sentences = []
for s in all_sentences:
    if not s:
        continue
    if EXCLUDE_SHORT and _word_count(s) < MIN_WORDS:
        continue
    if s not in seen:
        seen.add(s)
        unique_sentences.append(s)
all_sentences = unique_sentences

# Group sentences that differ by exactly one token
pattern_to_sentences = defaultdict(set)
sentence_tokens = {}

for s in all_sentences:
    toks = tokenize(s)
    sentence_tokens[s] = toks
    for i in range(len(toks)):
        pattern = tuple(toks[:i] + ["<*>"] + toks[i+1:])
        pattern_to_sentences[pattern].add(s)

groups = []
seen = set()

for pattern, sents in pattern_to_sentences.items():
    sents = sorted(sents)
    if len(sents) < 2:
        continue

    ok = True
    for i in range(len(sents)):
        for j in range(i + 1, len(sents)):
            a, b = sentence_tokens[sents[i]], sentence_tokens[sents[j]]
            if len(a) != len(b):
                ok = False
                break
            diffs = sum(x != y for x, y in zip(a, b))
            if diffs != 1:
                ok = False
                break
        if not ok:
            break

    if ok:
        key = tuple(sents)
        if key not in seen:
            seen.add(key)
            groups.append({"count": len(sents), "sentences": sents})

groups = sorted(groups, key=lambda x: (-x["count"], x["sentences"][0]))

print(f"Loaded {len(all_sentences)} unique sentences")
print(f"Found {len(groups)} groups differing by exactly one word/number\n")

for g in groups[:50]:
    print(f"[{g['count']}]")
    for s in g["sentences"]:
        print(f"  - {s}")
    print()


Loaded 1320 unique sentences
Found 55 groups differing by exactly one word/number

[7]
  - Air quality will probably become unhealthy on Friday.
  - Air quality will probably become unhealthy on Monday.
  - Air quality will probably become unhealthy on Saturday.
  - Air quality will probably become unhealthy on Sunday.
  - Air quality will probably become unhealthy on Thursday.
  - Air quality will probably become unhealthy on Tuesday.
  - Air quality will probably become unhealthy on Wednesday.

[5]
  - Clackamas County
  - Clark County
  - Columbia County
  - Multnomah County
  - Washington County

[4]
  - Evacuation Levels
  - Evacuation Order
  - Evacuation Preparedness
  - Evacuation Warning

[3]
  - A Wildfire Advisory has been issued for [location].
  - A Wildfire Warning has been issued for [location].
  - A wildfire watch has been issued for [location].

[3]
  - Emergency Preparedness
  - Evacuation Preparedness
  - PSPS Preparedness

[3]
  - Level 1
  - Level 2
  - Level 3

[